# OASIS Brain Mask Inspection

Run this notebook inside `py310sitreg_v1_test1` from `/homebase/code_sync/inrorspline`.

This audit treats raw `image > 0` as the canonical OASIS foreground mask and compares it against `seg > 0` as a label/comparison mask for the seed-3 shuffled first10 OASIS pairs.

In [ ]:
from pathlib import Path

REPO = Path('/homebase/code_sync/inrorspline')
PAIR_FILE = REPO / 'configs/splits/oasis/pair_test_200_shuffled_seed000003.txt'
OASIS_ROOT = Path('/database/wavereg/datasets/OASIS/OASIS_official/oasis_organized/split_v1')
OUT_DIR = REPO / 'outputs/data_inspection/oasis_mask_audit_seed3_first10'
REPO, PAIR_FILE.exists(), OASIS_ROOT.exists()

In [ ]:
!cd /homebase/code_sync/inrorspline && python scripts/data/inspection/oasis_mask_visual_audit.py \
  --pair-file configs/splits/oasis/pair_test_200_shuffled_seed000003.txt \
  --oasis-root /database/wavereg/datasets/OASIS/OASIS_official/oasis_organized/split_v1 \
  --num-pairs 10 \
  --out-dir outputs/data_inspection/oasis_mask_audit_seed3_first10

In [ ]:
import pandas as pd

summary_csv = OUT_DIR / 'oasis_mask_visual_audit_summary.csv'
oasis_df = pd.read_csv(summary_csv, dtype={'subject_id': str})
oasis_df.sort_values('dice')[[
    'subject_id',
    'orientation',
    'coronal_fixed_array_axis',
    'coronal_fixed_index',
    'coronal_aspect',
    'foreground_rule',
    'dice',
    'canonical_fg_fraction',
    'seg_label_fg_fraction',
    'seg_label_fg_covered_by_canonical_fg',
    'canonical_fg_covered_by_seg_label_fg',
    'figure',
]]

## Figure Review

Each panel shows raw image, canonical foreground `image > 0`, `seg > 0` as a comparator, and overlap/difference in standard coronal view. The display convention is top = patient superior, bottom = patient inferior, screen left = patient right, and screen right = patient left.

In [ ]:
from IPython.display import Image, Markdown, display

for row in oasis_df.sort_values('subject_id').itertuples(index=False):
    display(Markdown(f"### OASIS {row.subject_id} | Dice {row.dice:.4f}"))
    fig_path = Path(row.figure)
    if not fig_path.is_absolute():
        fig_path = REPO / fig_path
    display(Image(filename=str(fig_path), width=1100))